In [ ]:
import requests
import json
import time

auth = json.loads(mssparkutils.notebook.run("procore_auth"))
token = auth["token"]
COMPANY_ID = auth["company_id"]
headers = {
    "Authorization": f"Bearer {token}",
    "Procore-Company-Id": str(COMPANY_ID)
}

print("Auth successful" if token else "Auth failed")

StatementMeta(, e5efcefb-5bbb-40b7-86bc-75b75008d1a1, 6, Finished, Available, Finished, False)

Auth successful


In [5]:
projects_response = requests.get(
    "https://api.procore.com/rest/v1.0/projects",
    headers=headers,
    params={"company_id": COMPANY_ID}
)
projects = projects_response.json()
print(f"{len(projects)} projects found" if isinstance(projects, list) else "Failed to fetch projects")

StatementMeta(, e5efcefb-5bbb-40b7-86bc-75b75008d1a1, 7, Finished, Available, Finished, False)

16 projects found


In [6]:
all_cost_codes = []

for project in projects:
    project_id = project["id"]
    project_name = project["name"]
    print(f"Pulling cost codes for: {project_name}")

    page = 1
    while True:
        response = requests.get(
            "https://api.procore.com/rest/v1.0/cost_codes",
            headers=headers,
            params={
                "project_id": project_id,
                "page": page,
                "per_page": 100
            }
        )

        if response.status_code != 200:
            print(f"  Error {response.status_code} for {project_name}, skipping")
            break

        rows = response.json()

        if not rows or isinstance(rows, dict):
            break

        for row in rows:
            row["project_id"] = project_id
            row["project_name"] = project_name

        all_cost_codes.extend(rows)

        if len(rows) < 100:
            break

        page += 1
        time.sleep(0.3)

print(f"Done! Total cost codes: {len(all_cost_codes)}")

StatementMeta(, e5efcefb-5bbb-40b7-86bc-75b75008d1a1, 8, Finished, Available, Finished, False)

Pulling cost codes for: 1100 Fulton Street
Pulling cost codes for: 11 ESSEX ST
Pulling cost codes for: 337A & 337B West Broadway Rehabilitaion Work
Pulling cost codes for: 360 Lexington 8th & 20th Floor
Pulling cost codes for: 549 Munroe Av
Pulling cost codes for: 64 MET OVAL PSC + 1410 MET STOREROOM
Pulling cost codes for: Boys & Girls Club
Pulling cost codes for: EMBANKMENT PHASE II
Pulling cost codes for: Embankment + Revetment Apartments 270 & 310 10th Street NJ
Pulling cost codes for: Lillipvt 45 Renwick St
Pulling cost codes for: PCNA 711 11TH AVE
Pulling cost codes for: Sandbox Test Project
Pulling cost codes for: Standard Project Template
Pulling cost codes for: SYMRISE - 15th & 16th Flr
Pulling cost codes for: TEST - ABM SUBORDINATE
Pulling cost codes for: VOCO HOTEL TSQ
Done! Total cost codes: 4765


In [8]:
import pandas as pd
import re
import json

clean_rows = []
for row in all_cost_codes:
    clean_row = {}
    for key, value in row.items():
        if value is None:
            clean_row[key] = None
        elif isinstance(value, (dict, list)):
            clean_row[key] = json.dumps(value)
        elif isinstance(value, bool):
            clean_row[key] = str(value)
        elif isinstance(value, (int, float, str)):
            clean_row[key] = value
        else:
            clean_row[key] = str(value)  # catch anything else
    clean_rows.append(clean_row)

def clean_column_name(col):
    col = col.strip()
    col = re.sub(r'[ ,;{}()\n\t=]', '_', col)
    col = re.sub(r'_+', '_', col)
    col = col.strip('_')
    return col

pdf = pd.DataFrame(clean_rows)
pdf.columns = [clean_column_name(c) for c in pdf.columns]

# Convert any remaining object columns to string
for col in pdf.columns:
    if pdf[col].dtype == object:
        pdf[col] = pdf[col].astype(str).replace('None', None)

df = spark.createDataFrame(pdf)
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("procore_cost_codes_raw")

print("Saved to Bronze_Lakehouse successfully")

StatementMeta(, e5efcefb-5bbb-40b7-86bc-75b75008d1a1, 10, Finished, Available, Finished, True)

Saved to Bronze_Lakehouse successfully
